# Layerwise Intervention

Targeted interventions at specific layers: activation addition,
scaling experiments, direction interventions, cross-layer transfer,
and intervention sweeps.

## Why This Matters

Steering and intervention layerwise intervention enables controlled modification of model behavior by adding, removing, or modifying specific activation components. These causal interventions go beyond correlation to establish what internal representations actually do.

**Key references:**
- [Turner et al. (2023) "Activation Addition"](https://arxiv.org/abs/2308.10248) — Steering model behavior by adding vectors to activations
- [Li et al. (2023) "Inference-Time Intervention"](https://arxiv.org/abs/2306.03341) — Targeted activation interventions for truthfulness

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layerwise_intervention import (
    activation_addition,
    scaling_experiment,
    direction_intervention,
    cross_layer_transfer,
    intervention_sweep,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
direction = np.random.randn(cfg.d_model).astype(np.float32)
print("Model ready")

## Activation Addition

Add a direction vector to the residual stream at a specific layer.

In [ ]:
result = activation_addition(model, tokens, direction, layer=1, scale=0.5, metric_fn=metric_fn)
print(f"Metric change: {result['metric_change']:+.4f}")
print(f"\nTop promoted tokens:")
for t, v in result['top_promoted_tokens'][:3]:
    print(f"  Token {t}: {v:+.4f}")
print(f"Top demoted tokens:")
for t, v in result['top_demoted_tokens'][:3]:
    print(f"  Token {t}: {v:+.4f}")

## Scaling Experiment

In [ ]:
for layer in range(cfg.n_layers):
    result = scaling_experiment(model, tokens, layer=layer, metric_fn=metric_fn)
    print(f"Layer {layer}: sensitivity={result['sensitivity']:.4f}, optimal_scale={result['optimal_scale']:.2f}")

## Direction Intervention

Intervene with a direction at multiple layers and scales.

In [ ]:
result = direction_intervention(model, tokens, direction, metric_fn)
print(f"Best layer: {result['best_layer']}, best scale: {result['best_scale']:.1f}")
print(f"Best metric: {result['best_metric']:.4f}")
print(f"\nEffect profile by layer:")
for l in range(cfg.n_layers):
    print(f"  Layer {l}: max_effect={result['effect_profile'][l]:.4f}")

## Cross-Layer Transfer

In [ ]:
for src in range(cfg.n_layers):
    for tgt in range(cfg.n_layers):
        if src == tgt:
            continue
        result = cross_layer_transfer(model, tokens, src, tgt, metric_fn)
        print(f"L{src}->L{tgt}: change={result['metric_change']:+.4f}, cosine={result['cosine_similarity']:.3f}")

## Intervention Sweep

In [ ]:
result = intervention_sweep(model, tokens, metric_fn, n_directions=5)
print(f"Most sensitive layer: {result['most_sensitive_layer']}")
print(f"Least sensitive layer: {result['least_sensitive_layer']}")
for l in range(cfg.n_layers):
    print(f"  Layer {l}: sensitivity={result['layer_sensitivity'][l]:.4f}, "
          f"max_effect={result['layer_max_effect'][l]:.4f}")